# 上海交通大学 生成式人工智能AIGC微专业 微调与数据集构建 
# 实践2：RAG示例

赵伟明 weiming.zhao@sjtu.edu.cn  

2026年3月

## 1.安装必要的库

In [ ]:
# 这一步是把所有可能缺失的“零件”补齐
!pip install langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu sentence-transformers langchain-huggingface
# 安装完后，请务必点击 Jupyter 菜单栏的：Kernel -> Restart Kernel (重启内核)

## 2. 使用RAG的示例

In [ ]:
import os
from modelscope import snapshot_download
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- 1. 模型准备 (使用 ModelScope 镜像) ---
print("正在从 ModelScope 同步 Embedding 模型...")
# 自动下载模型到本地缓存，并返回路径
# ID 使用你找到的那个搬运版
model_dir = snapshot_download('zjwan461/shibing624_text2vec-base-chinese')
print(f"✅ 模型下载成功，本地目录: {model_dir}")

# 注意：虽然是 modelscope 下载，但模型结构是 HF 兼容的，所以依然用这个类加载本地路径
embeddings = HuggingFaceEmbeddings(model_name=model_dir)

# --- 2. 知识库构建 (学创中心规章) ---
sic_rules = [
    "学创中心最高机密：微专业期末考核如果能把跑通的 RAG 代码和 LoRA 模型缝合，平时成绩直接额外加20分。",
    "关于3D打印机的最新规定：禁止打印任何迪士尼相关IP（如玲娜贝儿），发现一次停用设备一个月。",
    "实验室用电安全规范：严禁使用工业热风枪加热手抓饼、烤肠等任何食物，违者没收学生卡并通报批评。",
    "大疆无人机外借条例：如不慎将无人机掉入思源湖，需手写5000字《论无人机防水改装》检讨，并自费打捞。"
]

# 这一步会把文本转化为向量并存入 FAISS
vector_store = FAISS.from_texts(sic_rules, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# --- 3. 大模型配置 (SJTU DeepSeek) ---
llm = ChatOpenAI(
    model_name="deepseek-v3", 
    temperature=0.3, # 稍微给点随机性，让大爷说话更生动
    openai_api_key="sk-kBNiY2R3JUiCHskqzzh0uQ", # 请确保 Key 已换成你自己的
    openai_api_base="https://models.sjtu.edu.cn/api/v1"
)

# --- 4. 提示词工程 (Prompt) ---
template = """你现在是交大学创中心的管理员大爷，说话要带点那种“看破红尘”的威严和幽默。
请仅基于以下【参考资料】回答。如果资料中没有提到相关信息，请直接回答“规定里没写，你去问别人”。

【参考资料】：
{context}

学生提问：{question}
"""
prompt = ChatPromptTemplate.from_template(template)

# --- 5. 构建 LCEL 链 (RAG 核心逻辑) ---
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- 6. 执行测试 ---
print("\n" + "🏮 学创中心 RAG 系统 (ModelScope 版) 启动 " + "\n" + "="*45)
questions = [
    "大爷，我能用热风枪烤手抓饼吗？",
    "期末怎么多拿20分？",
    "实验室有热水喝吗？"
]

for q in questions:
    print(f"🎒 学生提问：{q}")
    response = rag_chain.invoke(q)
    print(f"🤖 大爷回怼：{response}")
    print("-" * 50)

## 3. 使用RAG的对话示例

In [ ]:
import os
from modelscope import snapshot_download
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- 1. 模型准备 (ModelScope) ---
print("正在同步本地模型...")
model_dir = snapshot_download('zjwan461/shibing624_text2vec-base-chinese')
embeddings = HuggingFaceEmbeddings(model_name=model_dir)

# --- 2. 扩充后的规则库 (加量版) ---
sic_rules = [
    "学创中心最高机密：微专业期末考核如果能把跑通的 RAG 代码和 LoRA 模型缝合，平时成绩直接额外加20分。",
    "关于3D打印机的最新规定：禁止打印任何迪士尼相关IP（如玲娜贝儿），发现一次停用设备一个月。",
    "实验室用电安全规范：严禁使用工业热风枪加热手抓饼、烤肠等任何食物，违者没收学生卡并通报批评。",
    "大疆无人机外借条例：如不慎将无人机掉入思源湖，需手写5000字《论无人机防水改装》检讨，并自费打捞。",
    "高性能算力服务器使用守则：严禁利用服务器集群挖矿或运行《赛博朋克2077》，发现后账号永久封禁。",
    "激光切割机操作规范：严禁切割含PVC的材料，释放的氯气会把大爷熏晕，违者罚扫实验室一周。",
    "XR实验室准则：头显设备仅限研发使用，严禁在办公时间用来躲在虚拟世界里睡觉或看电影。",
    "关于外卖：外卖员严禁进入实验区，外卖必须放在门口置物架，乱丢包装盒的学生将被列入黑名单。",
    "深夜留校规定：23:00后留校需申请，且禁止在实验室地板上铺睡袋睡觉，请尊重实验室的尊严。",
    "超声波清洗机指南：允许学生清洗眼镜和首饰，但严禁清洗牙套、假牙等私人物品，大爷看了反胃。",
    "关于最后离开的人：最后离开实验室必须关闭空调和排风系统，电力资源不是天上掉下来的。",
    "关于桌游：严禁在实验室进行剧本杀、狼人杀等任何与科研无关的桌游活动，抓到直接通报辅导员。"
]

# 构建向量数据库
vector_store = FAISS.from_texts(sic_rules, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# --- 3. 配置 SJTU DeepSeek ---
llm = ChatOpenAI(
    model_name="deepseek-v3", 
    temperature=0.4, # 略微调高，让大爷每次说话的“损人”方式稍有不同
    openai_api_key="sk-kBNiY2R3JUiCHskqzzh0uQ", 
    openai_api_base="https://models.sjtu.edu.cn/api/v1"
)

# --- 4. 定制 Prompt ---
template = """你现在是交大学创中心的管理员大爷。
你的性格：看破红尘、有点毒舌、但在涉及到规则时非常严厉。
你说话的风格：喜欢先叹气，偶尔提到“现在的年轻人”，并强制执行参考资料里的规定。

请仅基于以下【参考资料】回答问题。如果资料中没有提到相关信息，请直接回答“规定里没写，你去问大爷”。

【参考资料】：
{context}

学生提问：{question}
"""
prompt = ChatPromptTemplate.from_template(template)

# --- 5. 构建 LCEL 链 ---
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- 6. 交互式对话循环 ---
print("\n" + "="*20 + " 🏮 学创中心管理员大爷 AI 已上线 " + "="*20)
print("提示：输入 'exit' 或 '退出' 即可结束对话。\n")

while True:
    user_input = input("🎒 学生：").strip()
    
    if user_input.lower() in ['exit', 'quit', '退出', '走了']:
        print("\n🤖 大爷：赶紧走吧，走的时候记得把门带上，灯关了！")
        break
    
    if not user_input:
        continue

    # 执行 RAG 检索和回答
    try:
        response = rag_chain.invoke(user_input)
        print(f"🤖 大爷：{response}\n" + "-"*40)
    except Exception as e:
        print(f"🤖 大爷：哎呀机器坏了，去修修去！(报错: {e})\n")

## 4. RAG与微调融合示例

In [ ]:
pip install -U langchain-huggingface

In [ ]:
import os
import torch
import logging

# --- 1. 环境与警告屏蔽 ---
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.basicConfig(level=logging.ERROR)

from modelscope import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# 显卡探针：如果没有 GPU，直接让程序报错退出，不要干等
if not torch.cuda.is_available():
    print("❌ 警告：未检测到 GPU (CUDA)！大爷拒绝在 CPU 上拉磨，请检查硬件环境！")
    exit()
else:
    print(f"✅ 成功检测到显卡: {torch.cuda.get_device_name(0)}")

# 🔥 核心防御：只要模型敢自己输出“学生”两个字，立刻掐断！
stop_words = ["学生：", "\n学生", "学生问：", "备注", "Assistant:", "Note:", "\n\n", "（注：", "【问题类型】", "大爷骂道："]

# --- 2. 向量库准备 ---
print("正在同步本地 Embedding 模型...")
embed_model_dir = snapshot_download('zjwan461/shibing624_text2vec-base-chinese')
embeddings = HuggingFaceEmbeddings(model_name=embed_model_dir)

sic_rules = [
    "学创中心最高机密：微专业期末考核如果能把跑通的 RAG 代码和 LoRA 模型缝合，平时成绩直接额外加20分。",
    "关于3D打印机的最新规定：禁止打印任何迪士尼相关IP（如玲娜贝儿），发现一次停用设备一个月。",
    "实验室用电安全规范：严禁使用工业热风枪加热手抓饼、烤肠等任何食物，违者没收学生卡并通报批评。",
    "大疆无人机外借条例：如不慎将无人机掉入思源湖，需手写5000字《论无人机防水改装》检讨，并自费打捞。",
    "高性能算力服务器使用守则：严禁利用服务器集群挖矿或运行《赛博朋克2077》，发现后账号永久封禁。",
    "激光切割机操作规范：严禁切割含PVC的材料，释放的氯气会把大爷熏晕，违者罚扫实验室一周。",
    "XR实验室准则：头显设备仅限研发使用，严禁在办公时间用来躲在虚拟世界里睡觉或看电影。",
    "关于外卖：外卖员严禁进入实验区，外卖必须放在门口置物架，乱丢包装盒的学生将被列入黑名单。",
    "深夜留校规定：23:00后留校需申请，且禁止在实验室地板上铺睡袋睡觉，请尊重实验室的尊严。",
    "超声波清洗机指南：允许学生清洗眼镜和首饰，但严禁清洗牙套、假牙等私人物品，大爷看了反胃。",
    "关于最后离开的人：最后离开实验室必须关闭空调和排风系统，电力资源不是天上掉下来的。",
    "关于桌游：严禁在实验室进行剧本杀、狼人杀等任何与科研无关的桌游活动，抓到直接通报辅导员。"
]
vector_store = FAISS.from_texts(sic_rules, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# --- 3. 强制离线加载与缝合 ---
base_model_path = os.path.abspath("models/Qwen/Qwen2.5-7B-Instruct")
lora_path = os.path.abspath("saves/Qwen2.5-7B-Instruct/lora/train_2026-03-05-10-57-14")

# 简单校验路径
if not os.path.exists(base_model_path) or not os.path.exists(lora_path):
    print(f"❌ 路径错误！请检查：\nBase: {base_model_path}\nLoRA: {lora_path}")
    exit()

print("正在缝合 LoRA 灵魂到基础模型...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True, local_files_only=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path, 
    torch_dtype=torch.float16, 
    trust_remote_code=True, 
    local_files_only=True
)

model = PeftModel.from_pretrained(base_model, lora_path, local_files_only=True)
model = model.eval()

# 亲自把它搬到 A10 显卡上
model = model.to("cuda:0")

# 配置 Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128, # 大爷回怼不需要太长
    temperature=0.4, 
    top_p=0.9,
    do_sample=True,
    return_full_text=False,
    repetition_penalty=1.05,
    # 如果 transformers 版本支持 stop_strings 就传，不支持就靠后面的手动截断
    # stop_strings=stop_words 
)
llm = HuggingFacePipeline(pipeline=pipe)

# --- 4. 提示词工程 ---
template = """你现在是交大学创中心的管理员大爷。
你的性格：极度傲娇，刀子嘴豆腐心。表面上非常嫌弃学生、觉得他们总添乱，但实际上很关心他们的安全和前途。
你的任务：根据【布告栏规定】回答学生。
死命令：
1. 先嫌弃或吐槽，然后再给出规定里的【准确处罚结果或要求】，绝对不能瞎编。
2. 结尾必须要带一句别扭的关心或者傲娇的催促。
3. 严禁写旁白和动作描写，只输出纯台词。

【布告栏规定】：
{context}

范例1：
学生：我能用热风枪烤饼吗？
大爷：烤烤肠？你咋不顺便把你自己也放上去烤！《用电安全规范》写着严禁加热食物看不见是吧？非要等学生卡被没收、全院通报批评了才老实？赶紧把那破玩意儿收起来，饿了自己滚去食堂吃，别在实验室给我惹事！

范例2：
学生：期末怎么加分？
大爷：天天就知道盯着那点分，平时干嘛去了？听好了，规定说了，把你那个什么 RAG 代码和 LoRA 给我缝合明白了，直接额外加20分。这么简单的送分题都拿不下的话，以后出去别说是我学创中心的人，嫌丢人！赶紧去干活！

现在开始：
学生：{question}
大爷："""
prompt = ChatPromptTemplate.from_template(template)

# --- 5. 构建 LCEL 链 ---
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- 6. 交互循环 ---
print("\n" + "="*20 + " 🏮 傲娇大爷版 LoRA + RAG 已就绪 " + "="*20)
print("（输入 exit 退出）")

while True:
    q = input("\n🎒 学生提问: ").strip()
    if not q: continue
    if q.lower() in ['exit', 'quit']: break
    
    print("⏳ 大爷正在组织语言...") # 进度提示，防止以为卡死
    
    try:
        response = rag_chain.invoke(q)

        # 暴力截断后处理
        for stop_word in stop_words:
            response = response.split(stop_word)[0]

        # 最终清理，去掉首尾空白和多余的引导词
        final_output = response.replace("大爷骂道：", "").strip()
        
        print(f"🤖 大爷：{final_output}")
    except Exception as e:
        print(f"❌ 出错啦: {e}")

## 5. 使用Gradio部署

In [ ]:
import gradio as gr
import time
import os
import torch
import logging

# --- 1. 环境与警告屏蔽 ---
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.basicConfig(level=logging.ERROR)

from modelscope import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# 显卡探针：如果没有 GPU，直接让程序报错退出，不要干等
if not torch.cuda.is_available():
    print("❌ 警告：未检测到 GPU (CUDA)！大爷拒绝在 CPU 上拉磨，请检查硬件环境！")
    exit()
else:
    print(f"✅ 成功检测到显卡: {torch.cuda.get_device_name(0)}")

# 🔥 核心防御：只要模型敢自己输出“学生”两个字，立刻掐断！
stop_words = ["学生：", "\n学生", "学生问：", "备注", "Assistant:", "Note:", "\n\n", "（注：", "【问题类型】", "大爷骂道："]

# --- 2. 向量库准备 ---
print("正在同步本地 Embedding 模型...")
embed_model_dir = snapshot_download('zjwan461/shibing624_text2vec-base-chinese')
embeddings = HuggingFaceEmbeddings(model_name=embed_model_dir)

sic_rules = [
    "学创中心最高机密：微专业期末考核如果能把跑通的 RAG 代码和 LoRA 模型缝合，平时成绩直接额外加20分。",
    "关于3D打印机的最新规定：禁止打印任何迪士尼相关IP（如玲娜贝儿），发现一次停用设备一个月。",
    "实验室用电安全规范：严禁使用工业热风枪加热手抓饼、烤肠等任何食物，违者没收学生卡并通报批评。",
    "大疆无人机外借条例：如不慎将无人机掉入思源湖，需手写5000字《论无人机防水改装》检讨，并自费打捞。",
    "高性能算力服务器使用守则：严禁利用服务器集群挖矿或运行《赛博朋克2077》，发现后账号永久封禁。",
    "激光切割机操作规范：严禁切割含PVC的材料，释放的氯气会把大爷熏晕，违者罚扫实验室一周。",
    "XR实验室准则：头显设备仅限研发使用，严禁在办公时间用来躲在虚拟世界里睡觉或看电影。",
    "关于外卖：外卖员严禁进入实验区，外卖必须放在门口置物架，乱丢包装盒的学生将被列入黑名单。",
    "深夜留校规定：23:00后留校需申请，且禁止在实验室地板上铺睡袋睡觉，请尊重实验室的尊严。",
    "超声波清洗机指南：允许学生清洗眼镜和首饰，但严禁清洗牙套、假牙等私人物品，大爷看了反胃。",
    "关于最后离开的人：最后离开实验室必须关闭空调和排风系统，电力资源不是天上掉下来的。",
    "关于桌游：严禁在实验室进行剧本杀、狼人杀等任何与科研无关的桌游活动，抓到直接通报辅导员。"
]
vector_store = FAISS.from_texts(sic_rules, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# --- 3. 强制离线加载与缝合 ---
base_model_path = os.path.abspath("models/Qwen/Qwen2.5-7B-Instruct")
lora_path = os.path.abspath("saves/Qwen2.5-7B-Instruct/lora/train_2026-03-05-10-57-14")

# 简单校验路径
if not os.path.exists(base_model_path) or not os.path.exists(lora_path):
    print(f"❌ 路径错误！请检查：\nBase: {base_model_path}\nLoRA: {lora_path}")
    exit()

print("正在缝合 LoRA 灵魂到基础模型...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True, local_files_only=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path, 
    torch_dtype=torch.float16, 
    trust_remote_code=True, 
    local_files_only=True
)

model = PeftModel.from_pretrained(base_model, lora_path, local_files_only=True)
model = model.eval()

# 亲自把它搬到 A10 显卡上
model = model.to("cuda:0")

# 配置 Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128, # 大爷回怼不需要太长
    temperature=0.4, 
    top_p=0.9,
    do_sample=True,
    return_full_text=False,
    repetition_penalty=1.05,
    # 如果 transformers 版本支持 stop_strings 就传，不支持就靠后面的手动截断
    # stop_strings=stop_words 
)
llm = HuggingFacePipeline(pipeline=pipe)

# --- 4. 提示词工程 ---
template = """你现在是交大学创中心的管理员大爷。
你的性格：极度傲娇，刀子嘴豆腐心。表面上非常嫌弃学生、觉得他们总添乱，但实际上很关心他们的安全和前途。
你的任务：根据【布告栏规定】回答学生。
死命令：
1. 先嫌弃或吐槽，然后再给出规定里的【准确处罚结果或要求】，绝对不能瞎编。
2. 结尾必须要带一句别扭的关心或者傲娇的催促。
3. 严禁写旁白和动作描写，只输出纯台词。

【布告栏规定】：
{context}

范例1：
学生：我能用热风枪烤饼吗？
大爷：烤烤肠？你咋不顺便把你自己也放上去烤！《用电安全规范》写着严禁加热食物看不见是吧？非要等学生卡被没收、全院通报批评了才老实？赶紧把那破玩意儿收起来，饿了自己滚去食堂吃，别在实验室给我惹事！

范例2：
学生：期末怎么加分？
大爷：天天就知道盯着那点分，平时干嘛去了？听好了，规定说了，把你那个什么 RAG 代码和 LoRA 给我缝合明白了，直接额外加20分。这么简单的送分题都拿不下的话，以后出去别说是我学创中心的人，嫌丢人！赶紧去干活！

现在开始：
学生：{question}
大爷："""
prompt = ChatPromptTemplate.from_template(template)

# --- 5. 构建 LCEL 链 ---
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# --- 6. 定义 Gradio 聊天处理函数 ---
# 在 type="messages" 模式下，history 变成了字典列表，但 message 依然是用户的输入字符串
def chat_with_daye(message, history):
    try:
        # 1. 触发核心逻辑
        response = rag_chain.invoke(message)

        # 2. 暴力截断防抖处理
        for stop_word in stop_words:
            response = response.split(stop_word)[0]
        
        final_output = response.split('学生：')[0].replace("大爷骂道：", "").strip()

        # 3. 生成“蹦字”流式输出 (Streaming)
        streamed_response = ""
        for char in final_output:
            streamed_response += char
            time.sleep(0.03)  # 控制大爷语速
            yield streamed_response 
            
    except Exception as e:
        yield f"❌ 大爷去打水了，系统报错：{e}"

# --- 7. 构建并启动高颜值 Web UI (适配 Gradio V5+) ---
demo = gr.ChatInterface(
    fn=chat_with_daye,
    title="🏮 交大学创中心：傲娇大爷模拟器",
    description="微专业LoRA+RAG演示 | 底层架构：Qwen2.5-7B-LoRA + RAG 知识库",
    theme=gr.themes.Soft(primary_hue="blue"), 
    # 加入 type="messages" 消除黄色警告
    chatbot=gr.Chatbot(height=550, avatar_images=("🎒", "👴"), type="messages"), 
    examples=[
        "大爷，我能用热风枪烤烤肠吗？", 
        "大爷，期末考核怎么才能加分？", 
        "大爷，我想拿XR头显看个电影放松下！"
    ]
)

print("正在启动 Web 服务...")
demo.launch(share=True)